# 03 - GOLD: Decision Layer + Intervention Simulator (Delta)

**Goal:** Convert Silver feature tables into a decision-ready Gold layer:
- Risk scoring (transparent, explainable)
- Risk bands for prioritization (Low / Medium / High)
- Business impact (expected revenue at risk)
- What-if simulator (expected recovered revenue under outreach assumptions)
- Data quality gates logged to `dq.check_results`

**Input:** `silver.model_features`  
**Outputs:**  
- `gold.fact_noshow_scoring` (Gold fact table)  
- `gold.v_fact_noshow_simulator` (scenario layer view)  
- DQ logs in `dq.check_results`



In [0]:
%sql
CREATE DATABASE IF NOT EXISTS gold;

## 1) Build Gold Fact Table (risk scoring + expected business impact)

### Important note (be honest)
This is a **rule-based risk score**, not a trained ML probability model.
If you want a true probability, you must calibrate it (or train a model and validate it).
We keep it explainable and decision-friendly for an ops demo.



In [0]:
%sql
CREATE OR REPLACE TABLE gold.fact_noshow_scoring
USING DELTA
AS
WITH scored AS (
  SELECT
    appointment_id,
    patient_id,
    appt_date,
    neighbourhood,
    no_show,

    lead_time_days,
    prev_noshow_rate,
    is_weekend,
    is_senior,

    /* Transparent risk score (bounded) */
    CAST(
      LEAST(
        0.95,
        0.10
        + 0.20 * CASE WHEN lead_time_days >= 30 THEN 1 ELSE 0 END
        + 0.25 * CASE WHEN prev_noshow_rate >= 0.5 THEN 1 ELSE 0 END
        + 0.10 * CASE WHEN is_weekend = 1 THEN 1 ELSE 0 END
        + 0.05 * is_senior
      ) AS DOUBLE
    ) AS risk_score
  FROM silver.model_features
)
SELECT
  appointment_id,
  patient_id,
  appt_date,
  neighbourhood,
  no_show,

  /* Keep the name if you want to plot it like a probability,
     but it's technically a score unless calibrated. */
  risk_score AS no_show_probability,

  CASE
    WHEN risk_score >= 0.45 THEN 'High'
    WHEN risk_score >= 0.25 THEN 'Medium'
    ELSE 'Low'
  END AS risk_band,

  /* Business assumptions (can be replaced with real billing later) */
  500 AS avg_revenue,

  /* Expected impact for THIS appointment */
  500 * risk_score AS expected_revenue_at_risk
FROM scored;

## 2) Quick Validation (existence + row count + sample)

These checks confirm the table was created and is populated.


In [0]:
%sql
SHOW TABLES IN gold;

SELECT COUNT(*) AS rows_in_gold
FROM gold.fact_noshow_scoring;

SELECT *
FROM gold.fact_noshow_scoring
LIMIT 10;


## 3) Data Quality Gates (Gold)

We log PASS/FAIL checks into `dq.check_results`.
These are minimum viable gates that catch common production failures:
- Null IDs
- Duplicate appointment_id check (count duplicate IDs)
- Probability/score out of range
- Null critical fields
- Non-trivial segmentation (not everything falling into one band)


In [0]:
%sql
-- 3.1 appointment_id NOT NULL
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_no_null_appointment_id',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'appointment_id must not be null'
FROM gold.fact_noshow_scoring
WHERE appointment_id IS NULL;

-- 3.2 patient_id NOT NULL
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_no_null_patient_id',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'patient_id must not be null'
FROM gold.fact_noshow_scoring
WHERE patient_id IS NULL;

-- 3.3 appt_date NOT NULL
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_no_null_appt_date',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'appt_date must not be null'
FROM gold.fact_noshow_scoring
WHERE appt_date IS NULL;

-- 3.4 appointment_id UNIQUE
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_unique_appointment_id',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'number of appointment_ids with duplicates'
FROM (
  SELECT appointment_id
  FROM gold.fact_noshow_scoring
  GROUP BY appointment_id
  HAVING COUNT(*) > 1
) d;

-- 3.5 probability/score bounds
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_probability_in_range',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'no_show_probability must be between 0 and 0.95'
FROM gold.fact_noshow_scoring
WHERE no_show_probability < 0 OR no_show_probability > 0.95 OR no_show_probability IS NULL;

-- 3.6 segmentation sanity: must have at least 2 bands present
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_risk_band_non_trivial',
  CASE WHEN COUNT(DISTINCT risk_band) >= 2 THEN 'PASS' ELSE 'FAIL' END,
  CASE WHEN COUNT(DISTINCT risk_band) >= 2 THEN 0 ELSE 1 END AS failed_count,
  CONCAT('distinct_bands=', COUNT(DISTINCT risk_band)) AS details
FROM gold.fact_noshow_scoring;

-- 3.7 neighbourhood NOT NULL
INSERT INTO dq.check_results
SELECT
  current_timestamp(),
  'gold_no_null_neighbourhood',
  CASE WHEN COUNT(*)=0 THEN 'PASS' ELSE 'FAIL' END,
  COUNT(*),
  'neighbourhood must not be null'
FROM gold.fact_noshow_scoring
WHERE neighbourhood IS NULL;

In [0]:
%sql
-- View latest gate results
SELECT *
FROM dq.check_results
WHERE check_name LIKE 'gold_%'
ORDER BY run_time_stamp DESC
LIMIT 25;


## 4) Sanity Checks (distribution + observed no-show rate by band)

This is the real credibility check:
If High-risk doesn’t actually have a higher observed no-show rate than Medium/Low,
your scoring is not useful (or features are broken / leaking).


In [0]:
%sql
-- score range
SELECT
  MIN(no_show_probability) AS min_prob,
  MAX(no_show_probability) AS max_prob,
  AVG(no_show_probability) AS avg_prob
FROM gold.fact_noshow_scoring;

-- band distribution
SELECT risk_band, COUNT(*) AS n
FROM gold.fact_noshow_scoring
GROUP BY risk_band
ORDER BY n DESC;

-- observed no-show rate by band
SELECT
  risk_band,
  COUNT(*) AS appointments,
  AVG(CAST(no_show AS DOUBLE)) AS observed_no_show_rate,
  AVG(no_show_probability) AS avg_score
FROM gold.fact_noshow_scoring
GROUP BY risk_band
ORDER BY avg_score DESC;


## 5) What-If Simulator (Scenario Layer via View)

We create a scenario layer **without changing the base Gold table**.
This supports experimenting with intervention assumptions.

**Lift interpretation:** if outreach reduces no-shows by X%, then expected recovered revenue:
`expected_revenue_at_risk * lift`

We allow lift to vary by risk band (more realistic than one constant).


In [0]:
%sql
CREATE OR REPLACE VIEW gold.v_fact_noshow_simulator AS
SELECT
  f.*,

  /* Scenario assumptions (tune as needed) */
  CASE
    WHEN risk_band = 'High' THEN 0.20
    WHEN risk_band = 'Medium' THEN 0.12
    ELSE 0.05
  END AS sms_lift_assumption,

  /* Expected recovered revenue */
  f.expected_revenue_at_risk *
  CASE
    WHEN risk_band = 'High' THEN 0.20
    WHEN risk_band = 'Medium' THEN 0.12
    ELSE 0.05
  END AS expected_recovered_revenue
FROM gold.fact_noshow_scoring f;


## 6) Simulator Validation (View)

Validate that the scenario layer produces non-null recovered revenue totals.


In [0]:
%sql
SELECT
  MIN(expected_recovered_revenue) AS min_rec,
  MAX(expected_recovered_revenue) AS max_rec,
  SUM(expected_recovered_revenue) AS sum_rec
FROM gold.v_fact_noshow_simulator;


## 7) Decision Outputs (for dashboard visuals)

### A) Priority targeting: recovered revenue by risk band
### B) Timing: monthly recovered revenue trend




In [0]:
%sql
-- A) Recovery impact by risk band
SELECT
  risk_band,
  COUNT(*) AS appointments,
  SUM(expected_revenue_at_risk) AS total_expected_revenue_at_risk,
  SUM(expected_recovered_revenue) AS total_expected_recovered_revenue
FROM gold.v_fact_noshow_simulator
GROUP BY risk_band
ORDER BY total_expected_recovered_revenue DESC;

-- B) Monthly recovery trend
SELECT
  YEAR(appt_date) AS year,
  MONTH(appt_date) AS month,
  SUM(expected_recovered_revenue) AS expected_recovered_revenue
FROM gold.v_fact_noshow_simulator
GROUP BY YEAR(appt_date), MONTH(appt_date)
ORDER BY year, month;


## 8) High-Risk Appointment Queue (Operational View)

Queue-style output a clinic ops team could act on.
Sorted by highest expected recovery first.


In [0]:
%sql
SELECT
  appointment_id,
  patient_id,
  neighbourhood,
  appt_date,
  no_show_probability,
  expected_revenue_at_risk,
  sms_lift_assumption,
  expected_recovered_revenue
FROM gold.v_fact_noshow_simulator
WHERE risk_band = 'High'
ORDER BY expected_recovered_revenue DESC
LIMIT 50;


## 9) Optional: Catalog / Schema sanity (only if you use workspace.* namespaces)

If your environment uses `workspace.gold` instead of `gold`, ensure you’re consistent.


In [0]:
%sql
SHOW TABLES IN gold;


## 10) Final KPI Summary (One-line Proof)

High-level totals for dashboard cards.


In [0]:
%sql
SELECT
  COUNT(*) AS total_appointments,
  AVG(CAST(no_show AS DOUBLE)) AS overall_no_show_rate,
  SUM(expected_revenue_at_risk) AS total_expected_revenue_at_risk,
  SUM(expected_recovered_revenue) AS total_expected_recovered_revenue
FROM gold.v_fact_noshow_simulator;

In [0]:
%sql
SELECT
  risk_band,
  COUNT(*) AS appointments,
  AVG(no_show_probability) AS avg_probability,
  SUM(expected_revenue_at_risk) AS total_revenue_at_risk,
  SUM(expected_recovered_revenue) AS total_expected_recovered_revenue,

  SUM(expected_recovered_revenue) / COUNT(*) 
    AS expected_recovery_per_appt

FROM gold.v_fact_noshow_simulator
GROUP BY risk_band
ORDER BY expected_recovery_per_appt DESC;


### Interpretation (how to act on this)

- **High risk** has the highest *severity* (avg no-show ≈ 0.56) and the best **unit ROI**: **$55.66 recovered per outreach** — ideal when staff bandwidth is limited (calls/SMS budget).
- **Medium risk** delivers the largest **total expected recovered revenue** (**$390K**) because it balances probability (0.33) with volume (19.5K) — best “primary” segment for scalable programs.
- **Low risk** shows high total revenue-at-risk (**$4.94M**) driven by **massive volume**, but outreach efficiency is poor (**$2.74 per outreach**) — only worth targeting with low-cost automation or if lift assumptions improve.

**Decision rule:**  
Start with **High** for highest ROI per contact, expand to **Medium** for maximum total recovery, and treat **Low** as automation-only.
